# Dataset

In [19]:
import torch
from torch.utils.data import Dataset
import librosa
import numpy as np
from transformers import AutoProcessor, AutoModel
import os

class MERTChunkDataset(Dataset):
    def __init__(self, audio_folder, chunk_length_sec=10.0, overlap_sec=5.0, target_sr=24000, device='cpu'):
        self.audio_folder = audio_folder
        self.chunk_length_sec = chunk_length_sec
        self.overlap_sec = overlap_sec
        self.chunk_stride = chunk_length_sec - overlap_sec
        self.target_sr = target_sr
        self.device = device

        self.processor = AutoProcessor.from_pretrained("m-a-p/MERT-v1-95M", trust_remote_code=True)
        self.model = AutoModel.from_pretrained("m-a-p/MERT-v1-95M", trust_remote_code=True)
        self.model.eval()
        self.model.to(device)

        self.data = []
        for path in os.listdir(audio_folder):
            self.data.extend(self._process_file(os.path.join(audio_folder, path)))

    def _process_file(self, path):
        y, sr = librosa.load(path, sr=self.target_sr)
        total_sec = y.shape[0] / sr
        chunk_starts = np.arange(0, total_sec, self.chunk_stride)
        file_chunks = []

        for start_sec in chunk_starts:
            end_sec = start_sec + self.chunk_length_sec
            start_sample = int(start_sec * sr)
            end_sample = int(end_sec * sr)
            chunk_audio = y[start_sample:end_sample].astype(np.float32)
            if len(chunk_audio) == 0:
                continue

            inputs = self.processor(raw_speech=chunk_audio, sampling_rate=sr, return_tensors="pt").to(self.device)
            with torch.no_grad():
                embeddings = self.model(**inputs).last_hidden_state.squeeze(0)

            file_chunks.append(embeddings.cpu())
        return file_chunks

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]


In [20]:
dataset = MERTChunkDataset("/home/saliherdemk/try_dataset/audio/", chunk_length_sec=10.0, overlap_sec=5.0, target_sr=24000, device='cpu')

print("Number of chunks:", len(dataset))
print("Shape of first chunk:", dataset[0].shape)


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [22]:
import os
import torch
import librosa
import numpy as np
from transformers import AutoProcessor, AutoModel
import json
from tqdm import tqdm

audio_folder = "/home/saliherdemk/try_dataset/audio/"
embedding_folder = "/home/saliherdemk/try_dataset//embeddings"
chunk_length_sec = 10.0
overlap_sec = 5.0
target_sr = 24000
device = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(embedding_folder, exist_ok=True)

processor = AutoProcessor.from_pretrained("m-a-p/MERT-v1-95M", trust_remote_code=True)
model = AutoModel.from_pretrained("m-a-p/MERT-v1-95M", trust_remote_code=True)
model.eval()
model.to(device)

audio_files = [f for f in os.listdir(audio_folder))]

for audio_file in tqdm(audio_files, desc="Processing audio"):
    audio_path = os.path.join(audio_folder, audio_file)
    beatmap_id = os.path.splitext(audio_file)[0]

    y, sr = librosa.load(audio_path, sr=target_sr)
    total_sec = y.shape[0] / sr
    chunk_stride = chunk_length_sec - overlap_sec
    chunk_starts = np.arange(0, total_sec, chunk_stride)

    for chunk_idx, start_sec in enumerate(chunk_starts):
        end_sec = start_sec + chunk_length_sec
        start_sample = int(start_sec * sr)
        end_sample = int(end_sec * sr)
        chunk_audio = y[start_sample:end_sample].astype(np.float32)
        if len(chunk_audio) == 0:
            continue

        inputs = processor(raw_speech=chunk_audio, sampling_rate=sr, return_tensors="pt").to(device)
        with torch.no_grad():
            embeddings = model(**inputs).last_hidden_state.squeeze(0).cpu()

        save_name = f"{beatmap_id}_chunk{chunk_idx}.pt"
        save_path = os.path.join(embedding_folder, save_name)
        torch.save(embeddings, save_path)



Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Processing audio:   2%|██▉                                                                                                                                                          | 1/54 [00:25<22:42, 25.71s/it]


KeyboardInterrupt: 

In [25]:
import pandas as pd

df = pd.read_csv("/home/saliherdemk/try_dataset/formatted.csv")

In [28]:
import pandas as pd
import numpy as np

chunk_length_sec = 10
overlap_sec = 5
chunk_length_ms = chunk_length_sec * 1000
chunk_stride_ms = (chunk_length_sec - overlap_sec) * 1000

def get_chunks_for_hit(hit_start_ms, duration_ms, chunk_length_ms, chunk_stride_ms):
    hit_end_ms = hit_start_ms + duration_ms
    chunk_starts = np.arange(0, hit_end_ms + chunk_stride_ms, chunk_stride_ms)
    chunk_ids = []
    for i, chunk_start in enumerate(chunk_starts):
        chunk_end = chunk_start + chunk_length_ms
        if hit_end_ms > chunk_start and hit_start_ms < chunk_end:
            chunk_ids.append(i)
    return chunk_ids

df["chunk_id"] = df.apply(lambda row: get_chunks_for_hit(row["time"], row["duration"], chunk_length_ms, chunk_stride_ms), axis=1)

df[df["id"] == "14309-0"].head(20)


,id,time,type,x,y,hit_sound,path,repeat,spinner_time,new_combo,...,volume,effects,difficulty_rating,meter,beat_length,mapper_id,beatmap_id,duration,delta_time,chunk_id
0,14309-0,295.0,circle,376,192,4,E|,0,0,False,...,60,0,5.22,4,352.941176,250808,14309,0,295,[0]
1,14309-0,647.0,slider,256,340,0,B|200:356|164:296,2,0,False,...,60,0,5.22,4,352.941176,250808,14309,176,352,[0]
2,14309-0,1353.0,circle,196,160,8,E|,0,0,False,...,60,0,5.22,4,352.941176,250808,14309,0,706,[0]
3,14309-0,1706.0,slider,340,272,0,B|376:256|400:292|440:272,1,0,True,...,60,0,5.22,4,352.941176,250808,14309,176,353,[0]
4,14309-0,2236.0,slider,172,112,0,B|136:128|112:92|72:112,1,0,False,...,60,0,5.22,4,352.941176,250808,14309,176,530,[0]
5,14309-0,2765.0,circle,256,192,8,E|,0,0,False,...,60,0,5.22,4,352.941176,250808,14309,0,529,[0]
6,14309-0,3118.0,circle,136,192,4,E|,0,0,True,...,60,0,5.22,4,352.941176,250808,14309,0,353,[0]
7,14309-0,3470.0,slider,256,44,0,B|312:28|348:88,2,0,False,...,60,0,5.22,4,352.941176,250808,14309,176,352,[0]
8,14309-0,4176.0,circle,316,224,8,E|,0,0,False,...,60,0,5.22,4,352.941176,250808,14309,0,706,[0]
9,14309-0,4353.0,circle,196,224,0,E|,0,0,False,...,60,0,5.22,4,352.941176,250808,14309,0,177,[0]


In [35]:
df_exploded = df.explode("chunk_id").reset_index(drop=True)

chunk_length_sec = 10
overlap_sec = 5
chunk_stride_sec = chunk_length_sec - overlap_sec
df_exploded['chunk_start'] = df_exploded['chunk_id'] * chunk_stride_sec * 1000
df_exploded['hit_start_rel'] = df_exploded['time'] - df_exploded['chunk_start']
df_exploded['hit_end_rel'] = df_exploded['hit_start_rel'] + df_exploded['duration']

grouped = df_exploded.groupby('chunk_id')

chunk_dict = {}
for chunk_id, group in grouped:
    chunk_rows = group.to_dict(orient='records')
    chunk_dict[chunk_id] = chunk_rows

chunk_dict[0]


[{'id': '14309-0',
  'time': 295.0,
  'type': 'circle',
  'x': 376,
  'y': 192,
  'hit_sound': 4,
  'path': 'E|',
  'repeat': 0,
  'spinner_time': 0,
  'new_combo': False,
  'slider_velocity': 1.9,
  'sample_set': 1,
  'volume': 60,
  'effects': 0,
  'difficulty_rating': 5.22,
  'meter': 4,
  'beat_length': 352.941176470588,
  'mapper_id': 250808,
  'beatmap_id': 14309,
  'duration': 0,
  'delta_time': 295,
  'chunk_id': 0,
  'chunk_start': 0,
  'hit_start_rel': 295.0,
  'hit_end_rel': 295.0},
 {'id': '14309-0',
  'time': 647.0,
  'type': 'slider',
  'x': 256,
  'y': 340,
  'hit_sound': 0,
  'path': 'B|200:356|164:296',
  'repeat': 2,
  'spinner_time': 0,
  'new_combo': False,
  'slider_velocity': 1.9,
  'sample_set': 1,
  'volume': 60,
  'effects': 0,
  'difficulty_rating': 5.22,
  'meter': 4,
  'beat_length': 352.941176470588,
  'mapper_id': 250808,
  'beatmap_id': 14309,
  'duration': 176,
  'delta_time': 352,
  'chunk_id': 0,
  'chunk_start': 0,
  'hit_start_rel': 647.0,
  'hit_end

In [47]:
display(df_exploded[df_exploded["id"] == "14309-0"][["id", "time", "type", "x", "y", "duration", "delta_time", "chunk_id", "chunk_start", "hit_start_rel", "hit_end_rel"]])


,id,time,type,x,y,duration,delta_time,chunk_id,chunk_start,hit_start_rel,hit_end_rel
0,14309-0,295.0,circle,376,192,0,295,0,0,295.0,295.0
1,14309-0,647.0,slider,256,340,176,352,0,0,647.0,823.0
2,14309-0,1353.0,circle,196,160,0,706,0,0,1353.0,1353.0
3,14309-0,1706.0,slider,340,272,176,353,0,0,1706.0,1882.0
4,14309-0,2236.0,slider,172,112,176,530,0,0,2236.0,2412.0
5,14309-0,2765.0,circle,256,192,0,529,0,0,2765.0,2765.0
6,14309-0,3118.0,circle,136,192,0,353,0,0,3118.0,3118.0
7,14309-0,3470.0,slider,256,44,176,352,0,0,3470.0,3646.0
8,14309-0,4176.0,circle,316,224,0,706,0,0,4176.0,4176.0
9,14309-0,4353.0,circle,196,224,0,177,0,0,4353.0,4353.0


In [30]:
grouped = df.groupby("id")


In [37]:
pd.options.display.max_rows = None

In [44]:
for idx, group in grouped:
    group = group.sort_values(by=["id", "chunk_id"])
    display(group[["id", "time", "type", "x", "y", "duration", "delta_time", "chunk_id", "chunk_start", "hit_start_rel", "hit_end_rel"]])
    # last_chunk_id = group.iloc[-1]["chunk_id"][-1]
    # for i in range(0, last_chunk_id + 1):
        
    break

,id,time,type,x,y,duration,delta_time,chunk_id,chunk_start,hit_start_rel,hit_end_rel
0,14309-0,295.0,circle,376,192,0,295,0,0,295.0,295.0
1,14309-0,647.0,slider,256,340,176,352,0,0,647.0,823.0
2,14309-0,1353.0,circle,196,160,0,706,0,0,1353.0,1353.0
3,14309-0,1706.0,slider,340,272,176,353,0,0,1706.0,1882.0
4,14309-0,2236.0,slider,172,112,176,530,0,0,2236.0,2412.0
5,14309-0,2765.0,circle,256,192,0,529,0,0,2765.0,2765.0
6,14309-0,3118.0,circle,136,192,0,353,0,0,3118.0,3118.0
7,14309-0,3470.0,slider,256,44,176,352,0,0,3470.0,3646.0
8,14309-0,4176.0,circle,316,224,0,706,0,0,4176.0,4176.0
9,14309-0,4353.0,circle,196,224,0,177,0,0,4353.0,4353.0
